# Étape 6 — Évaluation sur le test vierge

14 nouvelles offres, annotées avec le guide v3.1 **sans regarder aucune sortie de modèle**. Aucun modèle n'a été entraîné ni ajusté en les voyant.

Ce notebook **ne réentraîne rien** : il charge l'adaptateur v3 déjà sauvegardé et compare :
- le **modèle de base** (même prompt système que la v3) ;
- la **v3 brute**, **v3 + réparation**, **v3 + réparation + garde-fous** (le pipeline final).

**Avant de lancer :** Exécution → Modifier le type d'exécution → GPU T4. Durée : 20 à 30 min.

In [ ]:
# 1. Installation + Google Drive
!pip install -q transformers accelerate peft bitsandbytes pydantic pandas

from google.colab import drive
drive.mount('/content/drive')

import os
RACINE = '/content/drive/MyDrive/job-offer-extractor'
DOSSIER = f'{RACINE}/data'
RESULTATS = f'{RACINE}/resultats'
ADAPTER = f'{RACINE}/adapter_v3'
TEST_VIERGE = f'{DOSSIER}/test_vierge.jsonl'

assert os.path.exists(TEST_VIERGE), f'Place test_vierge.jsonl dans {DOSSIER}'
print('Adaptateur v3 sur Drive :', os.path.exists(ADAPTER))

In [ ]:
# 2. Connexion Hugging Face (utile seulement si l'adaptateur n'est pas sur Drive)
from huggingface_hub import login
login()

In [ ]:
# 3. Schéma et prompt système : EXACTEMENT ceux de la v3
import json, re
from typing import Literal, Optional
from pydantic import BaseModel, ValidationError

class Offre(BaseModel):
    poste: str
    entreprise: Optional[str]
    ville: Optional[str]
    type_contrat: Optional[Literal['PFE', 'stage', 'CDI', 'CDD', 'SIVP', 'freelance']]
    mode_travail: Optional[Literal['présentiel', 'hybride', 'remote']]
    duree_mois: Optional[float]
    remuneration: Optional[str]
    langue_offre: Literal['fr', 'en', 'ar']
    competences: list[str]

ORDRE = list(Offre.model_fields)
SCHEMA = json.dumps(Offre.model_json_schema(), ensure_ascii=False)

REGLES = '''Règles :
- poste = le titre de l'offre, sans "(H/F)" ni "F/H".
- type_contrat = "PFE" si l'offre parle de PFE ou de fin d'études, sinon "stage" pour un stage. Si plusieurs types sont cités, prendre le premier.
- remuneration = le montant s'il est donné ; "rémunéré" si l'offre dit seulement rémunéré ; une prime n'est pas un salaire ; "Confidentiel" = null.
- competences = tout ce qui s'écrirait dans la rubrique Compétences d'un CV : langages, frameworks, outils, logiciels, plateformes cloud, bases de données, méthodes, domaines techniques. Inclure aussi celles citées en exemple ou entre parenthèses.
- Ne sont PAS des compétences : tâches, concepts génériques (API seule, cache, scripts), secteurs d'activité, années d'expérience, diplômes, qualités humaines, noms d'entreprise seuls, technologies citées seulement dans la présentation de l'entreprise.
- Noms canoniques en alphabet latin, quelle que soit la langue de l'offre (Python, JavaScript). Séparer les compétences composées (Git/GitLab -> Git, GitLab), sauf CI/CD, TCP/IP, PL/SQL, Pub/Sub. "IA" s'écrit "intelligence artificielle". Chaque compétence apparaît une seule fois.
- langue_offre = UNE seule valeur, la langue dominante.
- ville et entreprise sont deux champs séparés.
- Si une information est absente, mets null. Ne devine jamais.'''

SYSTEM_PROMPT = (
    "Tu extrais les informations d'une offre d'emploi ou de stage. "
    'Réponds UNIQUEMENT avec un JSON valide respectant ce schéma :\n'
    + SCHEMA + '\n\n' + REGLES
)
print('Ordre des champs :', ORDRE)

In [ ]:
# 4. Charger le test vierge et vérifier qu'il ne recoupe pas les données d'entraînement
from collections import Counter

def lire(chemin):
    with open(chemin, encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

test = lire(TEST_VIERGE)
for ex in test:
    Offre.model_validate(ex['extraction'])
    ex['extraction'] = {k: ex['extraction'][k] for k in ORDRE}

anciens = lire(f'{DOSSIER}/offres_propres.jsonl')
urls_anciennes = {o.get('url') for o in anciens}
textes_anciens = {o['offre'][:300] for o in anciens}
communs = [ex['url'] for ex in test if ex['url'] in urls_anciennes or ex['offre'][:300] in textes_anciens]
assert not communs, f'Offres déjà présentes dans le dataset : {communs}'

print(f'{len(test)} offres, aucune en commun avec les 68 offres du dataset.')
print('Langues :', dict(Counter(e['extraction']['langue_offre'] for e in test)))
print('Contrats :', dict(Counter(e['extraction']['type_contrat'] for e in test)))

In [ ]:
# 5. Génération, réparation, garde-fous et scores (identiques au notebook 05)
import torch
import pandas as pd

def nettoyer(texte):
    return texte.strip().removeprefix('```json').removesuffix('```').strip()

@torch.no_grad()
def extraire(offre, max_new_tokens=900):
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': offre}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors='pt', return_dict=True
    ).to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

def reparer(brut):
    """Coupe la liste des compétences au premier doublon et referme le JSON.
    Fonctionne parce que `competences` est le dernier champ."""
    t = nettoyer(brut)
    i = t.find('"competences"')
    if i < 0:
        return t
    debut = t.index('[', i)
    items, vus = [], set()
    for m in re.finditer(r'"((?:[^"\\]|\\.)*)"', t[debut:]):
        try:
            item = json.loads('"' + m.group(1) + '"')
        except json.JSONDecodeError:
            break
        if item.strip().lower() in vus:
            break
        vus.add(item.strip().lower())
        items.append(item)
    return t[:debut] + json.dumps(items, ensure_ascii=False) + '}'

CHAMPS = ['poste', 'entreprise', 'ville', 'type_contrat', 'mode_travail',
          'duree_mois', 'remuneration', 'langue_offre']

def norm(v):
    if v is None:
        return None
    if isinstance(v, (int, float)):
        return round(float(v), 2)
    return ' '.join(str(v).lower().split())

def egal(champ, pred, vrai):
    if champ == 'duree_mois' and pred is not None and vrai is not None:
        try:
            return abs(float(pred) - float(vrai)) < 0.05
        except (TypeError, ValueError):
            return False
    return norm(pred) == norm(vrai)

GARDER_SLASH = {'ci/cd', 'tcp/ip', 'pl/sql', 'pub/sub', 'tests a/b', 'a/b'}

def norm_souple(v):
    if v is None:
        return None
    t = str(v).lower()
    t = re.sub(r'\(?\b(h/f|f/h)\b\)?', ' ', t)
    t = re.sub(r'[^\w\s#+./]', ' ', t)
    return ' '.join(t.split())

def eclater(competences):
    """Normalise et sépare sur « / », sauf CI/CD, TCP/IP, etc."""
    out = set()
    for c in competences or []:
        if not isinstance(c, str):
            continue
        n = norm_souple(c)
        if n in GARDER_SLASH:
            out.add(n)
        else:
            out.update(p.strip() for p in n.split('/') if p.strip())
    return out

def ancre(terme, texte):
    """Le terme apparaît-il dans l'offre comme mot entier ?"""
    motif = r'(?<!\w)' + re.escape(terme.strip()) + r'(?!\w)'
    return re.search(motif, texte, flags=re.IGNORECASE) is not None

def scorer(nom, sorties, reparation):
    lignes, tp, fp, fn = [], 0, 0, 0
    stp, sfp, sfn, n_pred, n_ancre = 0, 0, 0, 0, 0
    for ex, brut in zip(test, sorties):
        vrai = ex['extraction']
        ligne = {'url': ex.get('url'), 'langue': vrai['langue_offre'], 'sortie_brute': brut,
                 'repare': False}
        try:
            pred = json.loads(nettoyer(brut))
            ligne['json_valide'] = True
        except json.JSONDecodeError:
            ligne['json_valide'] = False
            pred = {}
            if reparation:
                try:
                    pred = json.loads(reparer(brut))
                    ligne['repare'] = True
                except (json.JSONDecodeError, ValueError):
                    pred = {}
        try:
            Offre.model_validate(pred)
            ligne['schema_ok'] = True
        except ValidationError:
            ligne['schema_ok'] = False
        for c in CHAMPS:
            ligne[c] = egal(c, pred.get(c), vrai.get(c))
        ligne['poste_souple'] = norm_souple(pred.get('poste')) == norm_souple(vrai.get('poste'))
        comps_pred = [x for x in (pred.get('competences') or []) if isinstance(x, str)]
        p = {norm(x) for x in comps_pred}
        v = {norm(x) for x in vrai['competences']}
        tp += len(p & v); fp += len(p - v); fn += len(v - p)
        ps, vs = eclater(comps_pred), eclater(vrai['competences'])
        stp += len(ps & vs); sfp += len(ps - vs); sfn += len(vs - ps)
        morceaux = [m.strip() for x in comps_pred
                    for m in ([x] if norm_souple(x) in GARDER_SLASH else x.split('/')) if m.strip()]
        n_pred += len(morceaux)
        n_ancre += sum(ancre(m, ex['offre']) for m in morceaux)
        lignes.append(ligne)
    df = pd.DataFrame(lignes)

    def prf(a, b, c):
        pr = a / (a + b) if a + b else 0
        rp = a / (a + c) if a + c else 0
        return pr, rp, (2 * pr * rp / (pr + rp) if pr + rp else 0)

    prec, rapp, f1 = prf(tp, fp, fn)
    sprec, srapp, sf1 = prf(stp, sfp, sfn)
    resume = {'JSON valide': df['json_valide'].mean(),
              'Réparées': df['repare'].mean(),
              'Schéma respecté': df['schema_ok'].mean()}
    resume.update({c: df[c].mean() for c in CHAMPS})
    resume['poste_souple'] = df['poste_souple'].mean()
    resume.update({'Compétences précision': prec, 'Compétences rappel': rapp, 'Compétences F1': f1,
                   'Compétences souples P': sprec, 'Compétences souples R': srapp,
                   'Compétences souples F1': sf1,
                   'Taux d’ancrage': n_ancre / n_pred if n_pred else 0,
                   'Compétences / offre': n_pred / len(df)})
    df.to_csv(f'{RESULTATS}/details_vierge_{nom}.csv', index=False)
    return pd.Series(resume, name=nom).round(3)

In [ ]:
# 6. Garde-fous (identiques au notebook 05)
def lire_pred(brut):
    try:
        return json.loads(nettoyer(brut))
    except json.JSONDecodeError:
        try:
            return json.loads(reparer(brut))
        except Exception:
            return {}

DUREE_RE = re.compile(
    r'\d+([.,]\d+)?\s*(mois|month|months|semaines?|weeks?|jours?|days?|ans?|years?)'
    r'|\d+\s*(يوما|يوم|أيام|أسابيع|أشهر|شهر)|يومين|شهرين',
    re.IGNORECASE)

def garde_fous(brut, offre):
    pred = lire_pred(brut)
    if not pred:
        return brut
    if pred.get('duree_mois') is not None and not DUREE_RE.search(offre):
        pred['duree_mois'] = None
    if isinstance(pred.get('remuneration'), str) and pred['remuneration'].strip().lower() in {'confidentiel', 'confidential'}:
        pred['remuneration'] = None
    return json.dumps(pred, ensure_ascii=False)

In [ ]:
# 7. Charger le modèle de base en 4 bits (même configuration que l'entraînement)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map='auto')
model.eval()

In [ ]:
# 8. Modèle de base sur le test vierge (environ 10 min)
sorties_base = []
for i, ex in enumerate(test):
    sorties_base.append(extraire(ex['offre']))
    print(f'base {i+1}/{len(test)}', end='\r')
res_base = scorer('base', sorties_base, reparation=False)
res_base

In [ ]:
# 9. Ajouter l'adaptateur v3 (sans réentraîner)
from peft import PeftModel

source = ADAPTER if os.path.exists(ADAPTER) else 'mayss1/qwen2.5-1.5b-offres-tn-lora'
model = PeftModel.from_pretrained(model, source)
model.eval()
print('Adaptateur chargé depuis', source)

In [ ]:
# 10. v3 sur le test vierge (environ 10 min)
sorties_v3 = []
for i, ex in enumerate(test):
    sorties_v3.append(extraire(ex['offre']))
    print(f'v3 {i+1}/{len(test)}', end='\r')

res_v3_brut = scorer('v3_brut', sorties_v3, reparation=False)
res_v3_repare = scorer('v3_repare', sorties_v3, reparation=True)
sorties_gf = [garde_fous(s, ex['offre']) for s, ex in zip(sorties_v3, test)]
res_v3_gf = scorer('v3_gardefous', sorties_gf, reparation=True)

In [ ]:
# 11. Tableau du test vierge
vierge = pd.concat([res_base.rename('base'), res_v3_brut.rename('v3 brut'),
                    res_v3_repare.rename('v3 réparé'), res_v3_gf.rename('v3 + garde-fous')], axis=1)
vierge.to_csv(f'{RESULTATS}/comparaison_test_vierge.csv')
print(f'Test vierge : {len(test)} offres (1 offre = {100/len(test):.1f} points)')
print('⚠️ « JSON valide » de la colonne garde-fous compte les sorties réécrites : utilise la valeur de « v3 brut ».')
vierge

In [ ]:
# 12. Comparaison avec le test d'origine (étiquettes v3)
origine = pd.read_csv(f'{RESULTATS}/comparaison_finale_v3.csv', index_col=0)
lignes = ['Schéma respecté', 'poste_souple', 'entreprise', 'ville', 'type_contrat', 'mode_travail',
          'duree_mois', 'remuneration', 'langue_offre', 'Compétences précision',
          'Compétences rappel', 'Compétences F1', 'Taux d’ancrage']
cmp = pd.DataFrame({
    'base (test d’origine)': origine.loc[lignes, 'base'],
    'base (test vierge)': vierge.loc[lignes, 'base'],
    'v3 pipeline (test d’origine)': origine.loc[lignes, 'v3 + garde-fous'],
    'v3 pipeline (test vierge)': vierge.loc[lignes, 'v3 + garde-fous'],
})
cmp.to_csv(f'{RESULTATS}/comparaison_origine_vs_vierge.csv')
cmp

In [ ]:
# 13. Analyse des erreurs de la v3 sur le test vierge
for ex, brut in zip(test, sorties_v3):
    pred, vrai = lire_pred(brut), ex['extraction']
    print('■', vrai['poste'][:70], f"({vrai['langue_offre']})")
    for c in CHAMPS:
        if not egal(c, pred.get(c), vrai.get(c)):
            print(f'   {c:13} attendu : {vrai.get(c)!r:35.35} | prédit : {pred.get(c)!r:.35}')
    p = {norm(x) for x in pred.get('competences', []) if isinstance(x, str)}
    v = {norm(x) for x in vrai['competences']}
    if p - v: print('   compétences en trop    :', sorted(p - v)[:10])
    if v - p: print('   compétences oubliées   :', sorted(v - p)[:10])